
# Notebook Overview

This notebook demonstrates Databricks SQL table operations, including:

- **Table Creation (CTAS):** Creating tables from existing data using `CREATE TABLE AS SELECT`.
- **Partitioning:** Creating partitioned tables for optimized queries.
- **Constraints:** Adding primary key, foreign key, and check constraints to enforce data integrity.
- **Cloning:** Using deep and shallow clones to duplicate tables with or without data.
- **Table Overwrite:** Refreshing table data using `CREATE OR REPLACE TABLE` and `INSERT OVERWRITE`.
- **Data Manipulation:** Inserting and updating rows to illustrate table independence and constraint enforcement.

Each section includes SQL examples and explanations to help understand Databricks table management features.

In [0]:
CREATE TABLE employees1 AS SELECT * FROM demo_employees;
SELECT * FROM employees1;

In [0]:
CREATE TABLE employees2 AS SELECT id as emp_id, name, 'demo' as dept FROM demo_employees;
SELECT * FROM employees2

In [0]:
CREATE TABLE employees3 PARTITIONED BY (Dept) AS SELECT * FROM employees2;
SELECT * FROM employees3

In [0]:
-- First, make columns NOT NULL before adding constraints
ALTER TABLE employees1 ALTER COLUMN id SET NOT NULL;
ALTER TABLE employees2 ALTER COLUMN emp_id SET NOT NULL;

-- Now add the constraints
ALTER TABLE employees1 ADD CONSTRAINT pk PRIMARY KEY(id);
ALTER TABLE employees2 ADD CONSTRAINT fk FOREIGN KEY (emp_id) REFERENCES employees1;
ALTER TABLE employees3 ADD CONSTRAINT deptname_check CHECK (Dept IN ('demo','finance'))

In [0]:
UPDATE employees3 SET dept = 'hr'

In [0]:
-- DEEP CLONE - copies all the data files. double storage cost. slow copy. fully independent
CREATE TABLE employees4 DEEP CLONE employees3;
SELECT * FROM employees4;

-- SHALLOW CLONE - references the same data files at source table while table creation. no extra storage cost. fast copy (only metatdata gets copied). not live linked to source
CREATE TABLE employees5 SHALLOW CLONE employees3;
SELECT * FROM employees5;

----------------- IMPORTANT NOTES -------------
-- **** In both cases data modification in the cloned table will not affect the source table ***
-- **** In both cases data modification at source table will not refelect in the cloned table ***


-- Insert a new row into employees3 (source table)
INSERT INTO employees3 VALUES (99, 'Alice', 'hr');

-- Check employees3 (should have 5 rows now)
SELECT 'employees3 (source)' as table_name, COUNT(*) as row_count FROM employees3
UNION ALL
-- Check employees4 (should have 4 rows - unaffected)
SELECT 'employees4 (deep clone)' as table_name, COUNT(*) as row_count FROM employees4
UNION ALL
-- Check employees5 (should have 4 rows - unaffected)
SELECT 'employees5 (shallow clone)' as table_name, COUNT(*) as row_count FROM employees5                

In [0]:
-- Insert a new row into employees3 (source table)
INSERT INTO employees3 VALUES (100, 'Beta', 'demo');

-- Check employees3 (should have 5 rows now)
SELECT 'employees3 (source)' as table_name, COUNT(*) as row_count FROM employees3
UNION ALL
-- Check employees4 (should have 4 rows - unaffected)
SELECT 'employees4 (deep clone)' as table_name, COUNT(*) as row_count FROM employees4
UNION ALL
-- Check employees5 (should have 4 rows - unaffected)
SELECT 'employees5 (shallow clone)' as table_name, COUNT(*) as row_count FROM employees5  

# Databricks SQL Table Overwrite: Advantages & Options

Table overwrite operations in Databricks SQL efficiently replace table contents without altering schema or metadata.  
Useful for refreshing data, correcting errors, or updating tables with new datasets.

## Available Options
1. **CREATE OR REPLACE TABLE**: Recreates the table, replacing data and schema.
2. **INSERT OVERWRITE**: Replaces all rows in the existing table with query results, preserving schema and constraints.

## Benefits
- Efficient data refresh without manual deletion.
- Preserves schema and constraints (with INSERT OVERWRITE).
- Simplifies ETL workflows and batch updates.
- Reduces risk of schema drift and accidental data loss.

## Examples
sql
CREATE OR REPLACE TABLE employees5 AS SELECT * FROM employees3;
INSERT OVERWRITE employees5 SELECT * FROM employees3;

In [0]:
-- Overwriting table or create a new table
CREATE OR REPLACE TABLE employees5 AS SELECT * FROM employees3;
SELECT * FROM employees5

In [0]:
-- Overwriting the existing table
-- safe to overwrite the table without modyfying existing schema
INSERT OVERWRITE employees5 SELECT * FROM employees3;
SELECT * FROM employees5